# 03 · Dataset 与 DataLoader：标签来自下一个单词

原项目返回 `(input, target, length)`，其中 target 是一个分类标签。
本项目返回 `(X, y)`，二者都是长度为 `seq_len` 的单词 ID 序列。
y 相对 X 右移一位，模型在每一个时间步都有一个训练目标。


In [ ]:
import os
import sys
from pathlib import Path

# 支持从项目根目录或 notebooks 目录启动。
# Kaggle 上可在这里显式设置：
# os.environ["TIME_MACHINE_PROJECT_ROOT"] = "/kaggle/working/Time_Machine"
hint = os.environ.get("TIME_MACHINE_PROJECT_ROOT")
if hint:
    candidates = [Path(hint).expanduser().resolve()]
else:
    candidates = []
    for parent in (Path.cwd(), *Path.cwd().parents):
        candidates.extend([parent / "Time_Machine", parent])
PROJECT_ROOT = next(
    (path for path in candidates
     if (path / "config.py").is_file() and (path / "src/generation.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("请在 Time_Machine 项目内启动，或设置 TIME_MACHINE_PROJECT_ROOT。")
existing_config = sys.modules.get("config")
if existing_config and Path(existing_config.__file__).resolve() != PROJECT_ROOT / "config.py":
    raise RuntimeError("kernel 中已经导入其他项目的 config，请重启 kernel 再运行本 Notebook。")
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from config import ExperimentConfig, get_device, resolve_data_path, resolve_output_dir

data_path = resolve_data_path()
output_dir = resolve_output_dir()
device = get_device("auto")
print("项目根目录:", PROJECT_ROOT)
print("数据文件:", data_path)
print("输出目录:", output_dir)
print("计算设备:", device)


## 1. 先用一段短文本理解 X / y

输入 `the time traveller was`，目标为 `time traveller was speaking`。
预测 `time` 时只可使用 `the`；预测 `traveller` 时可使用 `the time`，依此类推。


In [ ]:
from src.preprocessing import tokenize, build_vocab, decode
from src.dataset import TimeMachineDataset, prepare_data

toy_tokens = tokenize("the time traveller was speaking to us quietly today")
toy_vocab = build_vocab([toy_tokens], min_freq=1)
toy_dataset = TimeMachineDataset(toy_tokens, toy_vocab, seq_len=4)
print("窗口数量:", len(toy_dataset))
for index in range(len(toy_dataset)):
    X, y = toy_dataset[index]
    print(f"窗口 {index}:")
    print("X:", decode(X, toy_vocab))
    print("y:", decode(y, toy_vocab))


## 2. 正式数据准备

`prepare_data()` 整合读取、分词、顺序切分、训练词表和三个 DataLoader。
每个 split 独立创建窗口，窗口不会跨越训练 / 验证 / 测试边界。
最后不足一个完整窗口的少量目标会丢弃，统计中会列出数量。


In [ ]:
config = ExperimentConfig()
data = prepare_data(data_path, config)
vocab = data.vocab
print("总 token 数:", data.metadata["total_tokens"])
print("词表大小:", len(vocab))
display(pd.DataFrame(data.metadata["splits"]).T)


## 3. 查看真实样本的每个时间步

X、y 的 dtype 都是 `torch.long`。不再需要浮点数的 0/1 标签。
同一个窗口内 `X[1:] == y[:-1]`；最后一个目标是紧接窗口之后的单词。


In [ ]:
X, y = data.datasets["train"][0]
display(pd.DataFrame({
    "step": range(config.seq_len),
    "input_id": X.tolist(),
    "input_word": decode(X, vocab),
    "target_id": y.tolist(),
    "target_word": decode(y, vocab),
}))
print("X dtype:", X.dtype, "| y dtype:", y.dtype)
print("标签右移正确:", bool((X[1:] == y[:-1]).all()))


## 4. 从样本到 batch

默认 X 和 y 都是 `[batch_size, seq_len]`，即 `[32, 32]`。
完整窗口可以随机打乱，但窗口内部单词顺序不能打乱。
验证和测试不 shuffle；`drop_last=False` 保留最后不足一个 batch 的完整窗口。


In [ ]:
train_loader = data.loaders["train"]
valid_loader = data.loaders["valid"]
test_loader = data.loaders["test"]
batch_X, batch_y = next(iter(train_loader))
print("batch X:", batch_X.shape)
print("batch y:", batch_y.shape)
print("训练 / 验证 / 测试 batches:", len(train_loader), len(valid_loader), len(test_loader))
print("第一条输入:", " ".join(decode(batch_X[0], vocab)))
print("第一条目标:", " ".join(decode(batch_y[0], vocab)))


## 5. 与损失函数连接的形状

模型将输出 `[B,T,V]`：每个窗口、每个时间步、每个候选单词都有一个 logit。
`CrossEntropyLoss` 接收 `[B*T,V]` 和 `[B*T]`，y 为单词 ID，不需要 one-hot。
不先对 logits 调用 softmax，交叉熵内部会处理。


In [ ]:
B, T = batch_X.shape
V = len(vocab)
print("模型 logits 应为:", (B, T, V))
print("损失函数输入应为:", (B * T, V))
print("损失函数目标:", batch_y.reshape(-1).shape)
print("每个 batch 的目标 token 数:", batch_y.numel())


## 隐藏状态的约定与下一步

训练中的每个 batch 使用独立的固定长度窗口，隐藏状态从零开始，
不把上一批状态传给随机打乱的下一批。
文本生成时则保留状态，将新生成的单词逐步反馈给模型。

接下来分别运行 `04_rnn.ipynb`、`05_lstm.ipynb` 和 `06_gru.ipynb`。
每个 Notebook 都会重新准备数据，所以不依赖本 Notebook 的变量或迭代器状态。
